In [ ]:
from pathlib import Path
import pandas as pd

dict_ = {'Despesa':{'ano_de_compra': 2026, 'mês_de_compra': 'Fevereiro', 'tipo_pagamento': 'Pix', 'despesa': 'ovo', 'tipo_despesa': 'Alimentação', 'valor': 20, 'data': '2026-04-01', 'id_user': '0626d64d-be23-445d-81c2-101b6a036ec4'}}

In [11]:
df = pd.read_csv('NUBANK.csv',encoding='utf-8',sep=',')
df.rename(columns={'title':'Despesa','amount':'Valor','date':'Data'},inplace=True)

df['Tipo de Pagamento'] = 'Cartão de Crédito NUBANK'
df['Tipo de Despesa'] = ''

df['Pagamento'] = 'Á Vista'

for idx, row in df.iterrows():
    if row['Valor'] < 0 :
     df.drop(idx,axis=0,inplace=True)

df = df[['Data'] + ['Tipo de Pagamento'] + ['Despesa'] + ['Tipo de Despesa'] + ['Valor'] + ['Pagamento']]

df['Data'] = pd.to_datetime(df['Data'])


In [14]:
import re
categorias = {
'Alimentação':['comfrutas','amigao','panificadora'],
'Cabelereiro':['chies'],
'Cartão':[],
'Conhecimento':['asimov','unarrado','aprova total','estrategia','livro'],
'Diversos':['google one','notebook'],
'Farmácia':[],
'Igreja':[],
'Música':['música'],
'Plano Celular':['tim'],
'Presentes':[],
'Saúde':['h f','clinica''saúde','saude','treino','coluna','panobiancos'],
'Transporte':['partiu','postos','expresso','uber'],
}

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub('[^a-zA-ZÀ-ÿ0-9 ]','',texto)
    texto = texto.strip()
    
    return texto

def mapa_despesas():
   caminho = Path.cwd() / 'csv'
   #data_frames = []

   for csv in caminho.glob('*.csv'):
      map_df = pd.read_csv(csv,encoding='utf-8',sep=',')
    
   return map_df

In [15]:
meses = {
    1:'Janeiro',2:'Fevereiro',3:'Março',4:'Abril',5:'Maio',6:'Junho',
    7:'Julho',8:'Agosto',9:'Setembro',10:'Outubro',11:'Novembro',12:'Dezembro'
    }
df['Ano de Compra'] = df['Data'].dt.year
df['Mês de Compra'] = df['Data'].dt.month.map(meses)

map_df = mapa_despesas()
mapa = {
        limpar_texto(despesa): categoria 
        for despesa, categoria  in (zip(map_df['Despesa'],map_df['Tipo de Despesa']))}
    
def categorizar_despesas(despesa):
        despesa = limpar_texto(despesa)

        if despesa in mapa :
            return mapa[despesa]

        for categoria, tipos in categorias.items():
            for tipo in tipos:
                if tipo in despesa:
                    return categoria
                
        return 'Diversos'

df['Tipo de Despesa'] = df['Despesa'].apply(categorizar_despesas)
df['Valor'] = (df['Valor']
                        .astype(str)
                        .str.replace(',','.',regex=False)
                        .str.strip()
                        .astype(float)
                        )
    
df['Data'] = df['Data'].dt.strftime("%Y-%m-%d")
df.columns = df.columns.str.lower().str.strip()

df = df.rename(columns={'tipo de despesa':'tipo_despesa','tipo de pagamento':'tipo_pagamento','ano de compra':'ano_de_compra','mês de compra':'mês_de_compra'})